# NEXUS — Colab GPU + Ollama + Google Drive — modelos completos

Dias 1–5. Esta versão corrige o manifesto de modelos, o fallback do Ollama e mantém
modelos/cache/estado persistentes no Google Drive.

**Modelos Ollama:** `qwen3:1.7b`, `qwen3:4b`, `qwen3:8b`, `nomic-embed-text`.

**Dia 5 / Hugging Face:** `intfloat/multilingual-e5-small`,
`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`,
`nlptown/bert-base-multilingual-uncased-sentiment`,
`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`,
`deepset/xlm-roberta-base-squad2`, `Qwen/Qwen3-0.6B`.
O backend vLLM opcional do código usa `Qwen/Qwen3-8B`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os,sys,shutil,subprocess,time,json,re,uuid,importlib,platform,requests

D=Path("/content/drive/MyDrive/minicurso-mult-agents-colab")
DR=D/"repo"; DS=D/"state"; OM=D/"ollama/models"; HF=D/"huggingface"; RB=D/"rembg"
R=Path("/content/minicurso-mult-agents"); N=R/"codigo/nexus"
for p in (D,DS,OM,HF,RB): p.mkdir(parents=True,exist_ok=True)
def run(c,check=True,capture=False,env=None,cwd=None):
    return subprocess.run(c,shell=isinstance(c,str),check=check,text=True,capture_output=capture,env=env,cwd=cwd)
g=run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],False,True)
if g.returncode: raise RuntimeError("Selecione GPU no runtime do Colab")
print(g.stdout.strip())

In [ ]:
URL="https://github.com/overcyber/minicurso-mult-agents.git"
if not (DR/".git").exists(): run(["git","clone",URL,str(DR)])
elif not run(["git","-C",str(DR),"status","--porcelain"],capture=True).stdout.strip():
    run(["git","-C",str(DR),"pull","--ff-only","origin","main"])
if R.exists(): shutil.rmtree(R)
shutil.copytree(DR,R,ignore=shutil.ignore_patterns(".git","__pycache__","*.pyc",".chroma",".chroma_hf","*.db","saida","tracos"))
os.chdir(N)
cons=Path("/content/nexus-constraints.txt")
cons.write_text("langchain==1.4.0\nlanggraph==1.2.11\nlanggraph-checkpoint-sqlite==3.1.1\nlangchain-ollama==1.1.0\nlangchain-chroma==1.1.0\nchromadb==1.5.9\n")
run([sys.executable,"-m","pip","install","-q","-r",str(N/"requirements.txt"),"-r",str(N/"dia5/space_demo/requirements.txt"),"-c",str(cons),"accelerate>=1.0"])
os.environ.update(HF_HOME=str(HF),HF_HUB_CACHE=str(HF/"hub"),TRANSFORMERS_CACHE=str(HF/"transformers"),SENTENCE_TRANSFORMERS_HOME=str(HF/"sentence-transformers"),REMBG_HOME=str(RB),OLLAMA_MODELS=str(OM),OLLAMA_HOST="127.0.0.1:11434")

In [ ]:
OLLAMA={"small":"qwen3:1.7b","base":"qwen3:4b","medium":"qwen3:4b","large":"qwen3:8b","embedding":"nomic-embed-text"}
HFM={"embedding":"intfloat/multilingual-e5-small","reranker":"cross-encoder/mmarco-mMiniLMv2-L12-H384-v1","sentiment":"nlptown/bert-base-multilingual-uncased-sentiment","zero_shot":"MoritzLaurer/mDeBERTa-v3-base-mnli-xnli","qa":"deepset/xlm-roberta-base-squad2","chat":"Qwen/Qwen3-0.6B"}
VLLM_OPTIONAL="Qwen/Qwen3-8B"; REMBG_MODEL="u2net"
os.environ.update(MODELO=OLLAMA["base"],MODELO_PEQUENO=OLLAMA["small"],MODELO_MEDIO=OLLAMA["medium"],MODELO_GRANDE=OLLAMA["large"])
print("Ollama",OLLAMA); print("HF",HFM)

In [ ]:
def install_ollama():
    if shutil.which("ollama"): return
    run("apt-get update -qq",False); run("apt-get install -y -qq curl tar zstd ca-certificates")
    r=run("curl -fsSL https://ollama.com/install.sh | sh",False,True)
    if r.returncode==0 and shutil.which("ollama"): return
    arch={"x86_64":"amd64","amd64":"amd64","aarch64":"arm64","arm64":"arm64"}.get(platform.machine().lower())
    if not arch: raise RuntimeError("arquitetura sem fallback")
    u=f"https://ollama.com/download/ollama-linux-{arch}.tar.zst"
    r=run(f"curl -fsSL {u} | tar --zstd -x -C /usr",False,True)
    if r.returncode or not shutil.which("ollama"): raise RuntimeError(r.stdout+r.stderr)
install_ollama()
def alive():
    try:return requests.get("http://127.0.0.1:11434/api/tags",timeout=2).ok
    except:return False
if not alive():
    log=open(N/"ollama.log","ab",buffering=0)
    proc=subprocess.Popen(["ollama","serve"],env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)
    for _ in range(60):
        if alive():break
        time.sleep(1)
if not alive(): raise RuntimeError("Ollama não iniciou")

In [ ]:
def canon(x):
    x=(x or "").strip().lower()
    return x[:-7] if x.endswith(":latest") else x
def tags():
    return {m["name"] for m in requests.get("http://127.0.0.1:11434/api/tags",timeout=10).json().get("models",[])}
def pull(m):
    if canon(m) not in {canon(x) for x in tags()}: run(["ollama","pull",m],env=os.environ.copy())
for m in dict.fromkeys(OLLAMA.values()): pull(m)
need={canon(x) for x in OLLAMA.values()}
assert need <= {canon(x) for x in tags()}, need-{canon(x) for x in tags()}
print(sorted(tags()))
print(run(["ollama","run",OLLAMA["base"],"Responda apenas OK."],capture=True,env=os.environ.copy()).stdout)
print(run(["ollama","ps"],False,True,env=os.environ.copy()).stdout)

In [ ]:
from huggingface_hub import snapshot_download
for repo in dict.fromkeys(HFM.values()):
    print("[HF]",repo)
    snapshot_download(repo_id=repo,cache_dir=os.environ["HF_HUB_CACHE"],ignore_patterns=["onnx/*","openvino/*","*.tflite","*.h5","tf_model.*","flax_model.*"])
from rembg import new_session
_s=new_session(REMBG_MODEL); del _s
print("[OK] HF + rembg pré-carregados")

In [ ]:
STATE_DIRS=[".chroma",".chroma_hf",".chroma_hf_gpu","saida","tracos"]
def cp(a,b):
    if not a.exists(): return
    if a.is_dir():
        if b.exists(): shutil.rmtree(b)
        shutil.copytree(a,b)
    else:b.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(a,b)
def restore():
    for x in STATE_DIRS:cp(DS/x,N/x)
    for pat in ("*.db","*.db-wal","*.db-shm","*.csv","*.json","*.jsonl","ollama.log"):
        for a in DS.glob(pat):cp(a,N/a.name)
def persist():
    for x in STATE_DIRS:cp(N/x,DS/x)
    for pat in ("*.db","*.db-wal","*.db-shm","*.csv","*.json","*.jsonl","ollama.log"):
        for a in N.glob(pat):cp(a,DS/a.name)
restore()
MODS={"agente","clientes","ferramentas","indexar","nexus","hooks","steering","ferramentas_web","equipe","prompts","handoff","memoria_semantica","email_assistente","app_gradio","avaliacao","modelos","roteador","pipelines","embeddings_hf","cli"}
def dia(n):
    for m in list(sys.modules):
        if m in MODS:sys.modules.pop(m,None)
    ps=[str(N/f"dia{i}") for i in range(1,6)];sys.path[:]=[p for p in sys.path if p not in ps];sys.path.insert(0,str(N/n));importlib.invalidate_caches();os.chdir(N)

In [ ]:
# validação
run([sys.executable,"-m","compileall","-q",str(N)])
t=run([sys.executable,"-m","pytest","-q",str(N/"testes")],False,True,cwd=str(N));print(t.stdout)
if t.returncode:raise RuntimeError(t.stderr)
# auditoria literal dos modelos no código
patterns=[r"qwen3:[0-9.]+b",r"nomic-embed-text",r"Qwen/Qwen3-[0-9.]+B",r"intfloat/multilingual-e5-small",r"cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",r"nlptown/bert-base-multilingual-uncased-sentiment",r"MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",r"deepset/xlm-roberta-base-squad2"]
refs=set()
for py in N.rglob("*.py"):
    s=py.read_text(encoding="utf-8",errors="ignore")
    for p in patterns: refs.update(re.findall(p,s))
manifest=set(OLLAMA.values())|set(HFM.values())|{VLLM_OPTIONAL}
assert not(refs-manifest), refs-manifest
print("[OK] manifesto cobre referências:",sorted(refs))

# Dia 1

In [ ]:
dia("dia1");import ferramentas as f1;print(f1.calcular("950 * 24"));print(f1.ler_arquivo("../../etc/passwd"))
import agente as a1;print(a1.rodar("Qual foi o faturamento de 2024? Leia o documento necessário e cite o nome dele.",backend="ollama",verbose=True));persist()

# Dia 2

In [ ]:
dia("dia2");import indexar as i2
b=i2.construir(recriar=not(N/".chroma").exists())
for d in b.similarity_search("faturamento de 2024",k=2):print(Path(d.metadata.get("source","?")).name,d.page_content[:400])
import agente as a2;print(a2.rodar("Qual foi o faturamento de 2024? Cite a fonte."));persist()

# Dia 3

In [ ]:
dia("dia3");import hooks,steering
print(hooks.avaliar_politicas("ler_arquivo",{"caminho":"../../etc/passwd"},{}));print(steering.detectar_estagnacao({"passos":7,"achados":[]}))
from langchain_core.messages import HumanMessage
import nexus as n3
g=n3.compilar(str(N/"nexus_colab.db"),aprovar_ferramentas=False)
r=g.invoke({"messages":[HumanMessage("Qual foi o faturamento de 2024? Cite a fonte.")],"passos":0},{"configurable":{"thread_id":"colab-d3"},"recursion_limit":30})
print(r["messages"][-1].content);persist()

# Dia 4

In [ ]:
dia("dia4");from langchain_core.messages import HumanMessage;import equipe as e4
g=e4.compilar(str(N/"equipe_colab.db"));q="Compare os fornecedores e recomende um, citando fontes."
e={"messages":[HumanMessage(q)],"pergunta":q,"proximo":"","instrucao":q,"achados":[],"rascunho":"","veredito":"","rodadas":0}
r=g.invoke(e,{"configurable":{"thread_id":"colab-d4"},"recursion_limit":40});print(r.get("veredito"));print(r.get("rascunho"));persist()

# Dia 5

In [ ]:
import torch
dev="cuda" if torch.cuda.is_available() else "cpu";pdev=0 if torch.cuda.is_available() else -1
dia("dia5");import modelos as m5
exp={"supervisor":OLLAMA["small"],"pesquisador":OLLAMA["medium"],"analista":OLLAMA["small"],"redator":OLLAMA["large"],"critico":OLLAMA["medium"]}
for p,x in exp.items():print(p,m5.para(p).model);assert m5.para(p).model==x
from langchain_huggingface import HuggingFaceEmbeddings
emb=HuggingFaceEmbeddings(model_name=HFM["embedding"],model_kwargs={"device":dev},encode_kwargs={"normalize_embeddings":True});print("dim",len(emb.embed_query("faturamento 2024")))

In [ ]:
from transformers import pipeline
sent=pipeline("sentiment-analysis",model=HFM["sentiment"],device=pdev);print(sent(["Excelente","Péssimo"],truncation=True))
zs=pipeline("zero-shot-classification",model=HFM["zero_shot"],device=pdev);print(zs("Preciso comprar um torno industrial",candidate_labels=["maquinario agricola","equipamento industrial","servico"]))
qa=pipeline("question-answering",model=HFM["qa"],device=pdev)
ctx=(N/"dados/faq/politicas.md").read_text(encoding="utf-8");print(qa(question="Quanto tempo dura a garantia?",context=ctx))

In [ ]:
from sentence_transformers import CrossEncoder
rr=CrossEncoder(HFM["reranker"],device=dev);print("[OK] reranker",HFM["reranker"])
from transformers import AutoTokenizer,AutoModelForCausalLM
tok=AutoTokenizer.from_pretrained(HFM["chat"]);mdl=AutoModelForCausalLM.from_pretrained(HFM["chat"],torch_dtype="auto",device_map="auto")
print("[OK] chat",HFM["chat"],"device",mdl.device)
persist()

## Finalização

In [ ]:
persist()
print("Drive:",D)
print(run(["ollama","list"],False,True,env=os.environ.copy()).stdout)
print(run(["ollama","ps"],False,True,env=os.environ.copy()).stdout)